In [0]:
# ============================================================
#  Required fields cannot be null
# ============================================================

def test_required_fields_not_null():

    from pyspark.sql import functions as F

#Reads silver_shows table
    df = spark.table(
        "tvmaze.silver.silver_shows"
    )
#  Checks if table is empty
    row_count = df.count()
    assert row_count > 0, "Table is empty"

    null_counts = df.select(
        F.sum(F.col("show_id").isNull().cast("int")).alias("show_id_nulls"),
        F.sum(F.col("show_name").isNull().cast("int")).alias("show_name_nulls"),
        F.sum(F.col("runtime").isNull().cast("int")).alias("runtime_nulls")
    ).collect()[0]

    print("Null counts:", null_counts.asDict())

    assert null_counts["show_id_nulls"] == 0, f"show_id_nulls = {null_counts['show_id_nulls']}"
    assert null_counts["show_name_nulls"] == 0, f"show_name_nulls = {null_counts['show_name_nulls']}"
    assert null_counts["runtime_nulls"] == 0, f"runtime_nulls = {null_counts['runtime_nulls']}"
    

In [0]:
test_required_fields_not_null()

In [0]:
# ============================================================
#  Runtime cannot be less than zero
# ============================================================
def test_runtime():

    df = spark.table(
        "tvmaze.silver.silver_shows"
    )

    invalid_runtime = df.filter(df.runtime <= 0).count()
    print("Rows with runtime <= 0:", invalid_runtime)

    assert invalid_runtime == 0, f"Found {invalid_runtime} rows with runtime <= 0"

In [0]:
test_runtime()

In [0]:
# ============================================================
#  Unique show_id
# ============================================================
def test_unique_show_names_per_id():

    from pyspark.sql import functions as F
    df = spark.table("tvmaze.silver.silver_shows")
    
    # Finding duplicates
    duplicates = (
        df.groupBy("show_id")
          .count()
          .filter(F.col("count") > 1)
    )
    
    duplicates_cnt = duplicates.count()
    
    # Assert no duplicates
    assert duplicates_cnt == 0, f"Found {duplicates_cnt} duplicate show_ids"
    

In [0]:
test_unique_show_names_per_id()